# Desafio Motiva — Escadas rolantes (Dynamic Programming)

**Aluno:** Bruno Santos Castilho — RM 566799

Dataset sintético dos sensores das escadas rolantes da ViaQuatro (Linha 4-Amarela) usado em três atividades:
buscas, percurso na linha e plano de turno.

Regras seguidas: Python puro (só `csv`, `random` e `datetime` da biblioteca padrão), busca linear e binária
implementadas à mão, fila e pilha feitas com listas (`append`, `pop(0)`, `pop()`). Sem grafos, sem `deque`,
sem bibliotecas prontas de busca.

In [1]:
import csv
import random
from datetime import datetime, timedelta

SEMENTE = 42
N_ESCADAS = 60      # entre 30 e 197
N_LEITURAS = 240    # >= 200 (uma por hora = 10 dias)

# Faixas normais (mínimo, máximo) de cada sensor
FAIXAS = {
    "vibracao":    (1.0, 4.0),     # mm/s
    "temperatura": (35.0, 65.0),   # °C
    "corrente":    (10.0, 25.0),   # A
    "tensao":      (361.0, 399.0), # V
    "nivel_oleo":  (60.0, 100.0),  # %
    "velocidade":  (0.48, 0.52),   # m/s
}
SENSORES = list(FAIXAS)
CASAS = {"vibracao": 2, "temperatura": 1, "corrente": 1,
         "tensao": 1, "nivel_oleo": 1, "velocidade": 3}

# Linha 4-Amarela (ViaQuatro / Motiva), da base (Vila Sônia) até Luz
ESTACOES_L4 = ["Vila Sônia", "São Paulo-Morumbi", "Butantã", "Pinheiros",
               "Faria Lima", "Fradique Coutinho", "Oscar Freire", "Paulista",
               "Higienópolis-Mackenzie", "República", "Luz"]

INICIO_LEITURAS = datetime(2026, 9, 1, 0, 0)
COLUNAS = ["id_escada", "estacao", "grupo", "data_hora"] + SENSORES


def imprimir_tabela(cabecalho, linhas):
    """Imprime uma tabela simples alinhada em colunas."""
    larguras = [len(str(c)) for c in cabecalho]
    for linha in linhas:
        for i in range(len(linha)):
            if len(str(linha[i])) > larguras[i]:
                larguras[i] = len(str(linha[i]))
    fmt = " | ".join("{:<" + str(l) + "}" for l in larguras)
    print(fmt.format(*cabecalho))
    print("-+-".join("-" * l for l in larguras))
    for linha in linhas:
        print(fmt.format(*[str(x) for x in linha]))

## Etapa 0 — Dataset sintético

`gerar_dataset(n_escadas, n_leituras, semente)` usa `random.seed(semente)` e grava `sensores.csv`.

* **Saudável (~60%)**: todas as leituras dentro da faixa normal (ruído gaussiano em torno do centro da faixa).
* **Desgaste gradual (~30%)**: a partir de um ponto entre 30% e 60% da série, vibração, temperatura e corrente
  sobem e o nível de óleo cai de forma linear até o fim — os alertas aparecem e se acumulam no final.
* **Falhas pontuais (~10%)**: leituras normais com 4 a 8 picos isolados, cada um em um sensor sorteado.

Cada escada tem as leituras em ordem de horário (uma por hora); a ordem das escadas no arquivo é embaralhada.
O perfil não vai para o CSV (não está entre as colunas pedidas); a função o devolve à parte para o resumo.

In [2]:
def valor_normal(sensor):
    """Valor dentro da faixa normal, concentrado no centro."""
    lo, hi = FAIXAS[sensor]
    centro, meia = (lo + hi) / 2, (hi - lo) / 2
    v = random.gauss(centro, meia * 0.3)
    margem = meia * 0.05
    if v < lo + margem:
        v = lo + margem
    if v > hi - margem:
        v = hi - margem
    return v


def valor_falha(sensor):
    """Valor fora da faixa normal (pico de falha)."""
    if sensor == "vibracao":
        return random.uniform(5.0, 9.0)
    if sensor == "temperatura":
        return random.uniform(70.0, 85.0)
    if sensor == "corrente":
        return random.uniform(27.0, 35.0)
    if sensor == "tensao":
        return random.uniform(330.0, 355.0) if random.random() < 0.5 else random.uniform(405.0, 420.0)
    if sensor == "nivel_oleo":
        return random.uniform(35.0, 55.0)
    return random.uniform(0.42, 0.46) if random.random() < 0.5 else random.uniform(0.54, 0.58)


def gerar_dataset(n_escadas, n_leituras, semente, arquivo="sensores.csv"):
    if not (30 <= n_escadas <= 197):
        raise ValueError("n_escadas deve estar entre 30 e 197")
    if n_leituras < 200:
        raise ValueError("n_leituras deve ser pelo menos 200")
    random.seed(semente)

    # Perfis: 60% saudável, 30% desgaste, 10% falha pontual
    n_saud = round(n_escadas * 0.6)
    n_desg = round(n_escadas * 0.3)
    n_falha = n_escadas - n_saud - n_desg
    perfis_lista = ["saudável"] * n_saud + ["desgaste gradual"] * n_desg + ["falhas pontuais"] * n_falha
    random.shuffle(perfis_lista)

    ids = ["ESC-%03d" % (i + 1) for i in range(n_escadas)]
    perfis = {}
    escadas = []
    for i in range(n_escadas):
        perfis[ids[i]] = perfis_lista[i]
        escadas.append({
            "id": ids[i],
            "estacao": random.choice(ESTACOES_L4),
            "grupo": "crítica" if random.random() < 0.3 else "não crítica",
            "perfil": perfis_lista[i],
        })

    random.shuffle(escadas)  # escadas embaralhadas no arquivo

    linhas = []
    for esc in escadas:
        perfil = esc["perfil"]
        if perfil == "desgaste gradual":
            inicio_desg = random.randint(int(n_leituras * 0.3), int(n_leituras * 0.6))
            deriva = {"vibracao": random.uniform(3.0, 5.0),
                      "temperatura": random.uniform(12.0, 20.0),
                      "corrente": random.uniform(4.0, 8.0),
                      "nivel_oleo": -random.uniform(20.0, 35.0)}
        if perfil == "falhas pontuais":
            picos = {}
            for t in random.sample(range(n_leituras), random.randint(4, 8)):
                picos[t] = random.choice(SENSORES)

        for t in range(n_leituras):
            valores = {}
            for s in SENSORES:
                valores[s] = valor_normal(s)
            if perfil == "desgaste gradual" and t >= inicio_desg:
                p = (t - inicio_desg) / (n_leituras - 1 - inicio_desg)
                for s in deriva:
                    valores[s] += p * deriva[s]
            if perfil == "falhas pontuais" and t in picos:
                valores[picos[t]] = valor_falha(picos[t])

            linha = {"id_escada": esc["id"], "estacao": esc["estacao"], "grupo": esc["grupo"],
                     "data_hora": (INICIO_LEITURAS + timedelta(hours=t)).strftime("%Y-%m-%d %H:%M")}
            for s in SENSORES:
                linha[s] = round(valores[s], CASAS[s])
            linhas.append(linha)

    with open(arquivo, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=COLUNAS)
        w.writeheader()
        w.writerows(linhas)
    return linhas, perfis


_, PERFIS = gerar_dataset(N_ESCADAS, N_LEITURAS, SEMENTE)
print("sensores.csv gerado")

sensores.csv gerado


In [3]:
def ler_sensores(arquivo="sensores.csv"):
    leituras = []
    with open(arquivo, encoding="utf-8") as f:
        for linha in csv.DictReader(f):
            for s in SENSORES:
                linha[s] = float(linha[s])
            leituras.append(linha)
    return leituras


def em_alerta(leitura):
    """Uma leitura está em alerta quando algum sensor sai da faixa normal."""
    for s in SENSORES:
        lo, hi = FAIXAS[s]
        if leitura[s] < lo or leitura[s] > hi:
            return True
    return False


LEITURAS = ler_sensores()

# Resumo: mínimo, média e máximo por sensor
linhas_resumo = []
for s in SENSORES:
    minimo, maximo, soma = LEITURAS[0][s], LEITURAS[0][s], 0.0
    for l in LEITURAS:
        if l[s] < minimo: minimo = l[s]
        if l[s] > maximo: maximo = l[s]
        soma += l[s]
    linhas_resumo.append([s, FAIXAS[s], minimo, round(soma / len(LEITURAS), CASAS[s] + 1), maximo])

print(f"Leituras: {len(LEITURAS)}  |  Escadas: {N_ESCADAS}  |  Leituras por escada: {N_LEITURAS}  |  Semente: {SEMENTE}\n")
imprimir_tabela(["sensor", "faixa normal", "mínimo", "média", "máximo"], linhas_resumo)

# Escadas por perfil (com leituras em alerta de cada perfil)
contagem, alertas_perfil = {}, {}
for p in PERFIS.values():
    contagem[p] = contagem.get(p, 0) + 1
for l in LEITURAS:
    p = PERFIS[l["id_escada"]]
    if em_alerta(l):
        alertas_perfil[p] = alertas_perfil.get(p, 0) + 1
print()
imprimir_tabela(["perfil", "escadas", "% das escadas", "leituras em alerta"],
                [[p, contagem[p], f"{100 * contagem[p] / N_ESCADAS:.0f}%", alertas_perfil.get(p, 0)]
                 for p in ["saudável", "desgaste gradual", "falhas pontuais"]])

grupos = {}
ids_vistos = []
for l in LEITURAS:
    if l["id_escada"] not in ids_vistos:
        ids_vistos.append(l["id_escada"])
        grupos[l["grupo"]] = grupos.get(l["grupo"], 0) + 1
print("\nEscadas por grupo:", grupos)
print("Primeiras escadas na ordem do arquivo (embaralhadas):", ids_vistos[:8])

Leituras: 14400  |  Escadas: 60  |  Leituras por escada: 240  |  Semente: 42

sensor      | faixa normal   | mínimo | média  | máximo
------------+----------------+--------+--------+-------
vibracao    | (1.0, 4.0)     | 1.07   | 2.837  | 8.49  
temperatura | (35.0, 65.0)   | 35.8   | 51.21  | 81.4  
corrente    | (10.0, 25.0)   | 10.4   | 18.01  | 32.8  
tensao      | (361.0, 399.0) | 330.8  | 380.05 | 418.5 
nivel_oleo  | (60.0, 100.0)  | 35.8   | 77.67  | 99.0  
velocidade  | (0.48, 0.52)   | 0.449  | 0.5    | 0.57  

perfil           | escadas | % das escadas | leituras em alerta
-----------------+---------+---------------+-------------------
saudável         | 36      | 60%           | 0                 
desgaste gradual | 18      | 30%           | 1499              
falhas pontuais  | 6       | 10%           | 39                

Escadas por grupo: {'não crítica': 48, 'crítica': 12}
Primeiras escadas na ordem do arquivo (embaralhadas): ['ESC-050', 'ESC-010', 'ESC-044', 'ESC-052',

## Atividade 1 — Busca linear × busca binária

Contagem de comparações: cada teste entre o alvo e um elemento da lista conta 1 (na binária, até 2 por passo: `==` e `<`).

Para o 5º alerta, cada escada vira uma lista de **alertas acumulados** (0, 0, 1, 1, 2, ...). Ela nunca diminui e cresce
de 1 em 1, então a primeira posição com valor 5 é a leitura procurada. A busca linear acha essa posição varrendo do início;
a binária funciona porque a lista já está ordenada, e com `primeira=True` continua à esquerda até a primeira ocorrência.

In [4]:
def busca_linear(lista, alvo):
    """Devolve (posição, comparações). Posição -1 se não achar."""
    comparacoes = 0
    for i in range(len(lista)):
        comparacoes += 1
        if lista[i] == alvo:
            return i, comparacoes
    return -1, comparacoes


def busca_binaria(lista_ordenada, alvo, primeira=False):
    """Devolve (posição, comparações). A lista precisa estar ordenada.
    Com primeira=True continua à esquerda após achar, devolvendo a primeira ocorrência do alvo."""
    ini, fim, resposta, comparacoes = 0, len(lista_ordenada) - 1, -1, 0
    while ini <= fim:
        meio = (ini + fim) // 2
        comparacoes += 1
        if lista_ordenada[meio] == alvo:
            if not primeira:
                return meio, comparacoes
            resposta = meio
            fim = meio - 1
            continue
        comparacoes += 1
        if lista_ordenada[meio] < alvo:
            ini = meio + 1
        else:
            fim = meio - 1
    return resposta, comparacoes

In [5]:
# Lista de escadas na ordem do arquivo (embaralhada) e a mesma lista ordenada
ids_arquivo = []
info_escada = {}
leituras_por_escada = {}
for l in LEITURAS:
    if l["id_escada"] not in leituras_por_escada:
        ids_arquivo.append(l["id_escada"])
        info_escada[l["id_escada"]] = {"estacao": l["estacao"], "grupo": l["grupo"]}
        leituras_por_escada[l["id_escada"]] = []
    leituras_por_escada[l["id_escada"]].append(l)
ids_ordenados = sorted(ids_arquivo)

# 1.000 consultas por ID
random.seed(SEMENTE)
consultas = [random.choice(ids_arquivo) for _ in range(1000)]
comp_lin, comp_bin = [], []
for alvo in consultas:
    pos_l, c_l = busca_linear(ids_arquivo, alvo)
    pos_b, c_b = busca_binaria(ids_ordenados, alvo)
    assert ids_arquivo[pos_l] == alvo and ids_ordenados[pos_b] == alvo
    comp_lin.append(c_l)
    comp_bin.append(c_b)

def estat(v):
    return [sum(v), round(sum(v) / len(v), 2), min(v), max(v)]

tabela_id = [["linear (lista do arquivo)"] + estat(comp_lin),
             ["binária (lista ordenada)"] + estat(comp_bin)]
print(f"Busca por ID — {len(consultas)} consultas em {len(ids_arquivo)} escadas\n")
imprimir_tabela(["busca", "total", "média", "mínimo", "máximo"], tabela_id)
print(f"\nA binária fez {sum(comp_lin) / sum(comp_bin):.1f}x menos comparações que a linear.")

Busca por ID — 1000 consultas em 60 escadas

busca                     | total | média | mínimo | máximo
--------------------------+-------+-------+--------+-------
linear (lista do arquivo) | 30649 | 30.65 | 1      | 60    
binária (lista ordenada)  | 8950  | 8.95  | 1      | 11    

A binária fez 3.4x menos comparações que a linear.


In [6]:
# Primeira leitura em que os alertas acumulados chegam a 5
LIMITE_ALERTAS = 5
resultado_5 = []
for id_esc in ids_ordenados:
    leituras = leituras_por_escada[id_esc]
    acumulados, total = [], 0
    for l in leituras:
        if em_alerta(l):
            total += 1
        acumulados.append(total)
    pos_l, c_l = busca_linear(acumulados, LIMITE_ALERTAS)
    pos_b, c_b = busca_binaria(acumulados, LIMITE_ALERTAS, primeira=True)
    assert pos_l == pos_b
    resultado_5.append({"id_escada": id_esc, "estacao": info_escada[id_esc]["estacao"],
                        "grupo": info_escada[id_esc]["grupo"], "perfil": PERFIS[id_esc],
                        "total_alertas": total,
                        "leitura_5o_alerta": pos_l if pos_l >= 0 else "não atinge",
                        "data_hora": leituras[pos_l]["data_hora"] if pos_l >= 0 else "-",
                        "comp_linear": c_l, "comp_binaria": c_b})

imprimir_tabela(["escada", "perfil", "alertas", "leitura do 5º", "data_hora", "comp. linear", "comp. binária"],
                [[r["id_escada"], r["perfil"], r["total_alertas"], r["leitura_5o_alerta"],
                  r["data_hora"], r["comp_linear"], r["comp_binaria"]] for r in resultado_5])

atingem = [r for r in resultado_5 if r["data_hora"] != "-"]
cl = [r["comp_linear"] for r in resultado_5]
cb = [r["comp_binaria"] for r in resultado_5]
print()
imprimir_tabela(["busca (5 alertas acumulados)", "total", "média", "mínimo", "máximo"],
                [["linear"] + estat(cl), ["binária"] + estat(cb)])
print(f"\n{len(atingem)} de {len(resultado_5)} escadas chegam a 5 alertas acumulados.")

with open("comparacoes_buscas.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["teste", "busca", "total", "media", "minimo", "maximo"])
    w.writerow(["id_1000_consultas", "linear"] + estat(comp_lin))
    w.writerow(["id_1000_consultas", "binaria"] + estat(comp_bin))
    w.writerow(["5_alertas_por_escada", "linear"] + estat(cl))
    w.writerow(["5_alertas_por_escada", "binaria"] + estat(cb))
with open("alertas_5_por_escada.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(resultado_5[0]))
    w.writeheader()
    w.writerows(resultado_5)
print("Tabelas salvas em comparacoes_buscas.csv e alertas_5_por_escada.csv")

escada  | perfil           | alertas | leitura do 5º | data_hora        | comp. linear | comp. binária
--------+------------------+---------+---------------+------------------+--------------+--------------
ESC-001 | desgaste gradual | 71      | 166           | 2026-09-07 22:00 | 167          | 14           
ESC-002 | saudável         | 0       | não atinge    | -                | 240          | 16           
ESC-003 | falhas pontuais  | 7       | 105           | 2026-09-05 09:00 | 106          | 10           
ESC-004 | saudável         | 0       | não atinge    | -                | 240          | 16           
ESC-005 | saudável         | 0       | não atinge    | -                | 240          | 16           
ESC-006 | saudável         | 0       | não atinge    | -                | 240          | 16           
ESC-007 | falhas pontuais  | 7       | 201           | 2026-09-09 09:00 | 202          | 15           
ESC-008 | falhas pontuais  | 8       | 204           | 2026-09-09 12:00 |

## Atividade 2 — Percurso na linha com pilha

`linha.csv` tem as 11 estações da Linha 4-Amarela em ordem, com os minutos até a estação seguinte
(tempos estimados de percurso). A base de manutenção fica na primeira posição: **Vila Sônia**.

In [7]:
MINUTOS_PROXIMA = [2, 3, 2, 2, 2, 2, 2, 2, 2, 2, 0]  # última estação: 0

with open("linha.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["ordem", "estacao", "minutos_ate_proxima"])
    for i in range(len(ESTACOES_L4)):
        w.writerow([i + 1, ESTACOES_L4[i], MINUTOS_PROXIMA[i]])


def ler_linha(arquivo="linha.csv"):
    linha = []
    with open(arquivo, encoding="utf-8") as f:
        for r in csv.DictReader(f):
            linha.append({"estacao": r["estacao"], "minutos_ate_proxima": int(r["minutos_ate_proxima"])})
    return linha


def tempo_ate(linha, estacao):
    """Busca linear pela estação, somando os minutos desde a base (posição 0)."""
    minutos = 0
    for i in range(len(linha)):
        if linha[i]["estacao"] == estacao:
            return minutos
        minutos += linha[i]["minutos_ate_proxima"]
    raise ValueError(f"Estação não encontrada: {estacao}")


def rota_ida_e_volta(linha, estacao):
    """Ida: empilha as estações com append. Volta: desempilha com pop()."""
    pilha = []
    achou = False
    for i in range(len(linha)):
        pilha.append(linha[i]["estacao"])
        if linha[i]["estacao"] == estacao:
            achou = True
            break
    if not achou:
        raise ValueError(f"Estação não encontrada: {estacao}")
    ida = list(pilha)
    volta = []
    while pilha:
        volta.append(pilha.pop())
    minutos_ida = tempo_ate(linha, estacao)
    return {"ida": ida, "volta": volta, "minutos_ida": minutos_ida, "minutos_ida_volta": 2 * minutos_ida}


LINHA = ler_linha()
BASE = LINHA[0]["estacao"]

tempos = []
for e in LINHA:
    r = rota_ida_e_volta(LINHA, e["estacao"])
    tempos.append([e["estacao"], r["minutos_ida"], r["minutos_ida_volta"]])
with open("tempos_base.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["estacao", "minutos_ida", "minutos_ida_volta"])
    w.writerows(tempos)

print(f"Base: {BASE}\n")
imprimir_tabela(["estação", "minutos de ida", "minutos de ida e volta"], tempos)

exemplo = rota_ida_e_volta(LINHA, "Paulista")
print("\nExemplo — Paulista")
print("Ida:  ", " → ".join(exemplo["ida"]))
print("Volta:", " → ".join(exemplo["volta"]))

Base: Vila Sônia

estação                | minutos de ida | minutos de ida e volta
-----------------------+----------------+-----------------------
Vila Sônia             | 0              | 0                     
São Paulo-Morumbi      | 2              | 4                     
Butantã                | 5              | 10                    
Pinheiros              | 7              | 14                    
Faria Lima             | 9              | 18                    
Fradique Coutinho      | 11             | 22                    
Oscar Freire           | 13             | 26                    
Paulista               | 15             | 30                    
Higienópolis-Mackenzie | 17             | 34                    
República              | 19             | 38                    
Luz                    | 21             | 42                    

Exemplo — Paulista
Ida:   Vila Sônia → São Paulo-Morumbi → Butantã → Pinheiros → Faria Lima → Fradique Coutinho → Oscar Freire → Paulist

## Atividade 3 — Plano do turno com fila e pilha

* Escada **em alerta** = tem pelo menos uma leitura em alerta. O primeiro alerta é achado com
  `busca_binaria(acumulados, 1, primeira=True)`.
* `fila_critica` e `fila_normal` recebem as escadas na ordem do primeiro alerta (desempate pelo ID).
* A fila crítica é atendida antes da normal, retirando com `pop(0)`.
  Custo = ida + serviço (60 min crítica, 40 min normal) + volta. Cada atendimento sai da base e volta para ela.
* Se cabe no turno, `append` na agenda; se não cabe, vai para pendentes (e a próxima da fila é testada).
* Se o turno perde 1 hora (`planejar_turno(base, horas, perdeu_1h=True)`), a agenda é desfeita com `pop()` até caber;
  as escadas retiradas voltam para o início dos pendentes na ordem original.
* Turno começando às 08:00.

In [8]:
SERVICO = {"crítica": 60, "não crítica": 40}
INICIO_TURNO = "08:00"


def relogio(minutos, inicio=INICIO_TURNO):
    h, m = inicio.split(":")
    total = int(h) * 60 + int(m) + minutos
    return "%02d:%02d" % (total // 60, total % 60)


def escadas_em_alerta():
    lista = []
    for id_esc in ids_ordenados:
        acumulados, total = [], 0
        for l in leituras_por_escada[id_esc]:
            if em_alerta(l):
                total += 1
            acumulados.append(total)
        pos, _ = busca_binaria(acumulados, 1, primeira=True)
        if pos >= 0:
            lista.append({"id_escada": id_esc, "estacao": info_escada[id_esc]["estacao"],
                          "grupo": info_escada[id_esc]["grupo"],
                          "primeiro_alerta": leituras_por_escada[id_esc][pos]["data_hora"]})
    return sorted(lista, key=lambda e: (e["primeiro_alerta"], e["id_escada"]))


def montar_filas():
    fila_critica, fila_normal = [], []
    for e in escadas_em_alerta():
        if e["grupo"] == "crítica":
            fila_critica.append(e)
        else:
            fila_normal.append(e)
    return fila_critica, fila_normal


def planejar_turno(base, horas, perdeu_1h=False):
    if base != LINHA[0]["estacao"]:
        raise ValueError("A base precisa ser a primeira estação de linha.csv")
    limite = horas * 60
    fila_critica, fila_normal = montar_filas()
    agenda, pendentes = [], []
    usado = 0
    for fila in (fila_critica, fila_normal):
        while fila:
            esc = fila.pop(0)
            rota = rota_ida_e_volta(LINHA, esc["estacao"])
            custo = rota["minutos_ida"] + SERVICO[esc["grupo"]] + rota["minutos_ida"]
            item = dict(esc)
            item["custo_min"] = custo
            item["rota"] = " → ".join(rota["ida"] + rota["volta"][1:])
            if usado + custo <= limite:
                item["inicio_min"], item["fim_min"] = usado, usado + custo
                agenda.append(item)
                usado += custo
            else:
                pendentes.append(item)

    desfeitas = []
    if perdeu_1h:
        limite -= 60
        pilha = []                          # escadas retiradas da agenda
        while agenda and usado > limite:
            item = agenda.pop()
            usado -= item["custo_min"]
            pilha.append(item)
        while pilha:                        # desempilha para manter a ordem original
            desfeitas.append(pilha.pop())
        pendentes = desfeitas + pendentes

    return {"base": base, "horas": limite // 60, "agenda": agenda, "pendentes": pendentes,
            "minutos_usados": usado, "desfeitas": [d["id_escada"] for d in desfeitas]}


def mostrar_plano(plano, titulo):
    print(f"=== {titulo} — {plano['minutos_usados']} de {plano['horas'] * 60} min usados ===")
    imprimir_tabela(["#", "escada", "estação", "grupo", "início", "fim", "rota"],
                    [[i + 1, a["id_escada"], a["estacao"], a["grupo"], relogio(a["inicio_min"]),
                      relogio(a["fim_min"]), a["rota"]] for i, a in enumerate(plano["agenda"])])
    print(f"\nPendentes ({len(plano['pendentes'])}):",
          ", ".join(f"{p['id_escada']} ({p['grupo']})" for p in plano["pendentes"]) or "nenhum")
    print()

In [9]:
fc, fn = montar_filas()
print(f"Escadas em alerta: {len(fc) + len(fn)}  (fila crítica: {len(fc)}, fila normal: {len(fn)})\n")
imprimir_tabela(["fila", "escada", "estação", "primeiro alerta"],
                [["crítica", e["id_escada"], e["estacao"], e["primeiro_alerta"]] for e in fc] +
                [["normal", e["id_escada"], e["estacao"], e["primeiro_alerta"]] for e in fn])

Escadas em alerta: 24  (fila crítica: 4, fila normal: 20)

fila    | escada  | estação                | primeiro alerta 
--------+---------+------------------------+-----------------
crítica | ESC-008 | Fradique Coutinho      | 2026-09-01 11:00
crítica | ESC-003 | São Paulo-Morumbi      | 2026-09-01 22:00
crítica | ESC-023 | Fradique Coutinho      | 2026-09-02 04:00
crítica | ESC-033 | São Paulo-Morumbi      | 2026-09-06 21:00
normal  | ESC-043 | Faria Lima             | 2026-09-01 15:00
normal  | ESC-052 | Higienópolis-Mackenzie | 2026-09-01 16:00
normal  | ESC-007 | Paulista               | 2026-09-01 17:00
normal  | ESC-025 | Luz                    | 2026-09-05 10:00
normal  | ESC-040 | São Paulo-Morumbi      | 2026-09-05 14:00
normal  | ESC-049 | Higienópolis-Mackenzie | 2026-09-06 00:00
normal  | ESC-012 | República              | 2026-09-06 05:00
normal  | ESC-057 | Pinheiros              | 2026-09-06 07:00
normal  | ESC-038 | Higienópolis-Mackenzie | 2026-09-06 09:00
normal  | E

In [10]:
linhas_csv = []

def guardar(plano, turno, cenario):
    for i, a in enumerate(plano["agenda"]):
        linhas_csv.append([turno, cenario, "agenda", i + 1, a["id_escada"], a["estacao"], a["grupo"],
                           a["primeiro_alerta"], a["rota"], relogio(a["inicio_min"]), relogio(a["fim_min"]),
                           a["custo_min"]])
    for i, p in enumerate(plano["pendentes"]):
        linhas_csv.append([turno, cenario, "pendente", i + 1, p["id_escada"], p["estacao"], p["grupo"],
                           p["primeiro_alerta"], p["rota"], "-", "-", p["custo_min"]])

for horas in (4, 6, 8):
    plano = planejar_turno(BASE, horas)
    mostrar_plano(plano, f"Turno de {horas} h")
    guardar(plano, horas, "normal")

    reduzido = planejar_turno(BASE, horas, perdeu_1h=True)
    print(f"--- Turno de {horas} h perdeu 1 hora → desfeitas com pop(): {', '.join(reduzido['desfeitas']) or 'nenhuma'}")
    mostrar_plano(reduzido, f"Turno de {horas} h com 1 h a menos")
    guardar(reduzido, horas, "perdeu_1h")

with open("plano_turnos.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["turno_horas", "cenario", "situacao", "ordem", "id_escada", "estacao", "grupo",
                "primeiro_alerta", "rota", "inicio", "fim", "custo_min"])
    w.writerows(linhas_csv)
print("Plano salvo em plano_turnos.csv")

=== Turno de 4 h — 228 de 240 min usados ===
# | escada  | estação           | grupo   | início | fim   | rota                                                                                                                                                     
--+---------+-------------------+---------+--------+-------+----------------------------------------------------------------------------------------------------------------------------------------------------------
1 | ESC-008 | Fradique Coutinho | crítica | 08:00  | 09:22 | Vila Sônia → São Paulo-Morumbi → Butantã → Pinheiros → Faria Lima → Fradique Coutinho → Faria Lima → Pinheiros → Butantã → São Paulo-Morumbi → Vila Sônia
2 | ESC-003 | São Paulo-Morumbi | crítica | 09:22  | 10:26 | Vila Sônia → São Paulo-Morumbi → Vila Sônia                                                                                                              
3 | ESC-023 | Fradique Coutinho | crítica | 10:26  | 11:48 | Vila Sônia → São Paulo-Morumbi → B

--- Turno de 6 h perdeu 1 hora → desfeitas com pop(): ESC-043
=== Turno de 6 h com 1 h a menos — 292 de 300 min usados ===
# | escada  | estação           | grupo   | início | fim   | rota                                                                                                                                                     
--+---------+-------------------+---------+--------+-------+----------------------------------------------------------------------------------------------------------------------------------------------------------
1 | ESC-008 | Fradique Coutinho | crítica | 08:00  | 09:22 | Vila Sônia → São Paulo-Morumbi → Butantã → Pinheiros → Faria Lima → Fradique Coutinho → Faria Lima → Pinheiros → Butantã → São Paulo-Morumbi → Vila Sônia
2 | ESC-003 | São Paulo-Morumbi | crítica | 09:22  | 10:26 | Vila Sônia → São Paulo-Morumbi → Vila Sônia                                                                                                              
3 | ESC-023 | Fra